# Lab 23-03 · A DRAM Cell, Its Sense Amplifier, Refresh and Row Hammer

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import collections
import math
import re
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
import dramtech as dt


def ngspice(netlist):
    if shutil.which("ngspice") is None:
        print(f"ngspice is not installed: skipping {netlist} (the README lists the expected values)")
        return collections.defaultdict(lambda: float("nan"))
    out = subprocess.run(["ngspice", "-b", netlist], capture_output=True, text=True).stdout
    return {mm.group(1): float(mm.group(2)) for mm in re.finditer(r"^(\w+)\s*=\s*([-+0-9.eE]+)", out, re.M)}

## Part 1 · The cell in ngspice

*Predict* each value from §3.1, §3.3 and §3.4 first.

In [2]:
CS, CBL, K, VTH = 8e-15, 40e-15, 20e-6, 0.5
dv = dt.charge_sharing_dv(CS, CBL, 1.0, 0.5)
pred = {"dv_share": dv * 1e3,
        "vw1": dt.write_saturated(10e-9, K, CS, 1.0, VTH),
        "vw2": dt.write_saturated(10e-9, K, CS, 1.5, VTH),
        "def3_mv": dt.write_linear_deficit(10e-9, K, CS, 1.8, VTH, 1.0) * 1e3,
        "t_sense": dt.latch_time(40e-15 / 40e-6, 0.9, dv) * 1e9}
sim = ngspice("dram_cell.cir")
sim_t = sim.get("t_sense", float("nan")) * 1e9
for kk, vv in pred.items():
    got = sim_t if kk == "t_sense" else sim[kk]
    print(f"{kk:9s}: model {vv:.4f}, ngspice {got:.4f}")
print("units: mV for dv_share and def3_mv, V for vw1 and vw2, ns for t_sense (model = linear estimate)")
print(f"restored cell at 40 ns: {sim['snc_end']:.4f} V")

dv_share : model 83.3333, ngspice 83.3328
vw1      : model 0.4310, ngspice 0.4310
vw2      : model 0.9259, ngspice 0.9259
def3_mv  : model 0.2075, ngspice 0.2078
t_sense  : model 2.3795, ngspice 3.4625
units: mV for dv_share and def3_mv, V for vw1 and vw2, ns for t_sense (model = linear estimate)
restored cell at 40 ns: 1.0000 V


## Part 2 · Refresh overhead

*Predict* the 16 Gb overhead when the device runs hot.

In [3]:
dens = np.array([2, 4, 8, 16])
trfc = np.array([160, 260, 350, 350])
fig, ax = plt.subplots()
for refi, lab in ((7812.5, "up to 85 C (64 ms)"), (3906.25, "85-95 C (32 ms)")):
    ax.plot(dens, 100 * trfc / refi, "o-", label=lab)
ax.set_xscale("log", base=2); ax.set_xlabel("device density (Gb)"); ax.set_ylabel("refresh overhead (%)")
ax.legend(); plt.show()
for d_, t_ in zip(dens, trfc):
    print(f"{d_:2d} Gb: {100*dt.refresh_overhead(t_, 7812.5):.2f}% normal, {100*dt.refresh_overhead(t_, 3906.25):.2f}% hot")

 2 Gb: 2.05% normal, 4.10% hot
 4 Gb: 3.33% normal, 6.66% hot
 8 Gb: 4.48% normal, 8.96% hot
16 Gb: 4.48% normal, 8.96% hot


## Part 3 · The weak-cell tail

*Predict*: does halving the window remove half the weak cells, or far more?

In [4]:
N, MED, SIG = 2**34, 10.0, 1.0
win = np.logspace(np.log10(0.004), 0, 60)
weak = [dt.weak_cells(N, w, MED, SIG) for w in win]
plt.figure(); plt.loglog(win * 1e3, np.maximum(weak, 1e-3))
plt.axhline(1, ls="--"); plt.xlabel("refresh window (ms)"); plt.ylabel("expected failing cells"); plt.show()
print(f"64 ms: {dt.weak_cells(N, 0.064, MED, SIG):.0f} cells; 32 ms: {dt.weak_cells(N, 0.032, MED, SIG):.1f} cells")
t1 = MED * math.exp(SIG * stats.norm.ppf(1 / N))
print(f"window for under one expected weak cell: {t1*1e3:.1f} ms")

64 ms: 3766 cells; 32 ms: 79.1 cells
window for under one expected weak cell: 15.9 ms


## Part 4 · PARA Monte Carlo

*Predict* the miss fraction for p = 0.002 and 2,000 activations before running.

In [5]:
p, nth, trials = 0.002, 2000, 20_000
rng = np.random.default_rng(2303)
refreshes = rng.binomial(nth, p / 2, trials)  # refreshes of the victim during one attack
mc = np.mean(refreshes == 0)
exact = dt.para_miss_probability(p, nth)
print(f"Monte Carlo {mc:.4f}, exact {exact:.4f}, standard error {math.sqrt(exact*(1-exact)/trials):.4f}")
n = np.logspace(2, 5, 40)
plt.figure(); plt.loglog(n, [dt.para_p_for(1e-15, int(x)) for x in n])
plt.xlabel("row-hammer threshold (activations)"); plt.ylabel("PARA probability p for 1e-15 misses"); plt.show()

Monte Carlo 0.1388, exact 0.1352, standard error 0.0024
